# Thesis: Reclaimed Timber in Deep Generative Design
**Notebook ID:** 11_12_15_main_geometry_generation  
**Author:** Jasper Cluistra   
**Last Updated:** 2026-03-03
### Properties script
**Goal:** Main script that generates the geometry dataset that acts as input for the structural analysis in grasshopper, this script can take several variables as input at generates a CSV file made up of properties that can generate this digital geometry to a CAD geometry   
**Inputs:**   
*   Define base mesh
*   Allowed movement for different vertices
*   divisions over allowed movement

**Outputs:**

*   CSV file, with sample id, vertices, coordinates and edges

# 1.1 IMPORTING AND SETTING PARAMETERS

In [ ]:
import pandas as pd
import numpy as np
import c11_params
import config

# 1.2 GEOMETRY GENERATION

In [ ]:
from geometry import get_corner_indices

# --- TESTEN ---
grid_2x2 = get_corner_indices(2, 2)
print(f"2x2 Grid Indices: {grid_2x2}")

grid_3x3 = get_corner_indices(3, 3)
print(f"3x3 Grid Indices: {grid_3x3}")

grid_4x4 = get_corner_indices(4, 4)
print(f"4x4 Grid Indices: {grid_4x4}")
# Verwacht: 0 (BL), 4 (BR), 20 (TL), 24 (TR) -> want 5 punten per rij

# In je loop:
corners = get_corner_indices(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)
corner_values = corners.values() # [0, 4, 20, 24] bijv.
print(corner_values)

## Calculating Vertices

In [ ]:
from geometry import generate_sample_vertices, get_valid_shifts

def generate_full_dataset(num_samples):
    """Maakt de gigantische random dataset aan."""
    valid_shifts = get_valid_shifts(c11_params.DIVISIONS, c11_params.EDGE_LENGTH)
    all_data = []
    
    for sample_id in range(num_samples):
        # We geven params=None mee, dus hij genereert randoms
        vertices = generate_sample_vertices(sample_id, params=None, valid_shifts=valid_shifts)
        all_data.extend(vertices)
        
    return pd.DataFrame(all_data)

## Executing and saving

In [ ]:
from geometry import generate_edges
from config import RAW_DATA_PATH

# --- EXECUTE ---
final_vertices = generate_full_dataset(c11_params.NUM_SAMPLES)
final_edges = generate_edges(c11_params.NUM_SAMPLES, c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y)

# --- SAVE ---
final_vertices.to_csv(RAW_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_vertices.csv", index=False)
final_edges.to_csv(RAW_DATA_PATH / f"dataset_{c11_params.GRID}_{c11_params.NUM_SAMPLES}_edges.csv", index=False)

print(f"Generatie gereed voor {c11_params.NUM_SAMPLES} samples.")
print(f"Grid grootte: {c11_params.GRID}")

print("Vertices Shape:", final_vertices.shape)
print("Edges Shape:", final_edges.shape)

print(f"\nTotaal aantal regels in vertices CSV: {len(final_vertices)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_vertices.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_vertices[final_vertices['sample_id'] == 1].head(5))

print(f"\nTotaal aantal regels in edges CSV: {len(final_edges)}")
print("\nVoorbeeld output (eerste 5 regels van sample 0):")
print(final_edges.head(5))
print("\nVoorbeeld output (eerste 5 regels van sample 1):")
print(final_edges[final_edges['sample_id'] == 1].head(5))

# 1.5 SEARCH SPACE

In [ ]:
import json
from data_io import define_search_space

# --- UITVOEREN EN BEKIJKEN ---

# Gebruik de variabelen uit je eerdere configuratie (bijv. 1x1 grid)
ai_search_space = define_search_space(c11_params.GRID_CELLS_X, c11_params.GRID_CELLS_Y, c11_params.DIVISIONS, c11_params.EDGE_LENGTH)

print(f"De Search Space bevat in totaal {len(ai_search_space)} onafhankelijke variabelen (knoppen waar de AI aan mag draaien).")
print("\nVoorbeeld van hoe de computer dit ziet:")

# Toon de eerste 5 variabelen mooi geformatteerd
for var_name, bounds in list(ai_search_space.items()):
    print(f"- {var_name}: {bounds}")

In [7]:
# Sla de dictionary op als een JSON bestand
json_path = 'search_space.json'
with open(json_path, 'w') as f:
    json.dump(ai_search_space, f, indent=4) # indent=4 maakt het mooi leesbaar

print(f"✅ Search Space succesvol opgeslagen als '{json_path}'")

✅ Search Space succesvol opgeslagen als 'search_space.json'
